In [ ]:
import os 
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.document_loaders import WebBaseLoader

from dotenv import load_dotenv
load_dotenv()

USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [2]:
# data source and document
url = "https://www.bdstall.com/details/grip-strength-trainer-set-with-counter-adjustable-hand-146041/"
loader = WebBaseLoader(url)
docs = loader.load()

In [4]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=os.getenv("GOOGLE_API_KEY"))

In [5]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="test"
)

In [ ]:
# convert vector store into a retriever 
# retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

retriever = vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 3, "lambda_mult": 1} # k is number of top results, lambda_mult is diversity balance value, 1 is similar typical similarity search, 0 is very diverse. 
)

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k': 5}),
    llm=ChatGoogleGenerativeAI(model="models/gemini-2.5-flash")
)

## Contextual Compression Retrieval 

In [8]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash")
compressor = LLMChainExtractor.from_llm(llm)

compressor_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

ModuleNotFoundError: No module named 'langchain.retrievers'

In [7]:
query = "What strength training?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n-----Result {i + 1} -----")
    print(doc.page_content)